1. evaluate a bunch of CNNs with a couple of different metanetworks
2. get intra-CNN variance to analyze how metanetwork-invariant the results are
3. if this is the case, see if we can cluster bad vs. good CNNs for unlearning
4. Analyze why this is

# 0. Train a couple of metanetworks

In [ ]:
from cnn_surgery.utils.load_dataset import load_multi_stage_dataset
from cnn_surgery.lenses.regressor_lens import get_regressor_lens
import torch
from tqdm import tqdm
import os
import json

datasets = ["mnist", "fashion_mnist", "cifar10"]

for dataset in datasets:
    train, val, _ = load_multi_stage_dataset(include_test=False, dataset=dataset).values()

    weights_train = train[0]
    weights_val = val[0]

    accuracies_train = train[1]
    accuracies_val = val[1]

    configs_train = train[2]
    configs_val = val[2]

    for i in range(5):
        MetaNetwork, metrics = get_regressor_lens(
            weights_train,
            accuracies_train,
            weights_val,
            accuracies_val,
            device="cpu",
            return_metrics=True,
            verbose=False,
        ) # type: ignore

        # make sure parent directory exists
        os.makedirs("../models/good_bad_experiment_2", exist_ok=True)
        torch.save(MetaNetwork.state_dict(), f"../models/good_bad_experiment_2/{dataset}_metanetwork_{i}.pt")

        print(metrics)

        # save metrics
        metrics_dict = {
            "mse_train": metrics[0][0],
            "mae_train": metrics[0][1],
            "mse_val": metrics[1][0],
            "mae_val": metrics[1][1],
            "r2_val": metrics[2],
        }

        with open(f"../models/good_bad_experiment_2/{dataset}_metanetwork_{i}_metrics.json", "w") as f:
            json.dump(metrics_dict, f, indent=2)

# 1. Evaluate a bunch of CNNs with these metanetworks